# Prototype – does removing the leading silence change the AASIST score?

This notebook runs the prototype step by step: it selects recordings from ASVspoof 2019 LA eval
and removes the silence from their beginning.

**How to use it**

- Run the cells from top to bottom.
- The code of every step is in `src/xai_audio_deepfake_shortcuts_audit/` (one function per file);
  the notebook only calls the functions in the right order.
- The notebook only **reads** the dataset – nothing in `datasets/` is changed.

**Steps**

1. Setup – paths and settings
2. Select the recordings
3. Remove the leading silence
4. Listen and check

**Next:** the trimmed recordings go to the pod to be processed by AASIST (`scripts/prototype/score_aasist.py`).

### Step 1 – Setup: paths and settings

**What it does:** Loads the libraries, finds the project folder and sets the paths to the
ASVspoof 2019 LA eval protocol and audio files, to the outputs folder for working files
(`outputs/06_prototype_leading_silence_output`) and to the results folder
(`results/prototype`). It also sets all parameters of the prototype in one place: how many
recordings to select, the random seed and the silence detection settings, including the
thresholds to compare by listening in Step 4.

**Why:** All the next cells use these paths and settings. Keeping the parameters here makes it
clear with which values the results were produced, and they can be changed in one place.

In [ ]:
from pathlib import Path

from IPython.display import display

from xai_audio_deepfake_shortcuts_audit.prototype.select_files import select_files


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("pyproject.toml not found")


REPO_ROOT = find_repo_root(Path.cwd())
LA_DIR = REPO_ROOT / "datasets" / "01_asvspoof_2019" / "LA"
PROTOCOL_PATH = LA_DIR / "ASVspoof2019_LA_cm_protocols" / "ASVspoof2019.LA.cm.eval.trl.txt"
EVAL_AUDIO_DIR = LA_DIR / "ASVspoof2019_LA_eval" / "flac"

NOTEBOOK_NAME = "06_prototype_leading_silence"
OUTPUTS_DIR = REPO_ROOT / "outputs" / f"{NOTEBOOK_NAME}_output"
RESULTS_DIR = REPO_ROOT / "results" / "prototype"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Selection of recordings
N_PER_ATTACK = 3  # spoofed per attack (13 attacks); the same number of bona fide in total
SEED = 42  # random seed: the same seed gives the same selection

# Silence detection (find_speech_start / trim_leading_silence)
SAMPLE_RATE = 16000  # Hz
FRAME_MS = 25.0  # frame length
HOP_MS = 10.0  # step between frames
THRESHOLDS_DB = [-30.0, -35.0, -40.0]  # thresholds compared by listening in Step 4 (at most 3)
THRESHOLD_DB = -30.0  # chosen after listening in Step 4; also used for the model


def trimmed_dir(threshold_db: float) -> Path:
    """Folder with the recordings trimmed at this threshold, e.g. trimmed_-40dB."""
    return OUTPUTS_DIR / f"trimmed_{threshold_db:+.0f}dB"


print("Protocol file:", PROTOCOL_PATH.relative_to(REPO_ROOT), "| exists:", PROTOCOL_PATH.exists())
print("Audio folder: ", EVAL_AUDIO_DIR.relative_to(REPO_ROOT), "| exists:", EVAL_AUDIO_DIR.exists())

### Step 2 – Select the recordings

**What it does:** Calls `select_files`, which randomly selects `N_PER_ATTACK` spoofed recordings
from each of the 13 attacks (A07–A19) and the same total number of bona fide recordings. Then it shows the table, the number of recordings per class
and per attack, and checks that every selected file exists on disk.

**Why:** To select a small dataset for the prototype, that is reproducible.

In [ ]:
selection = select_files(PROTOCOL_PATH, n_per_attack=N_PER_ATTACK, seed=SEED)

print("Selected recordings:", len(selection))
display(selection)

In [ ]:
print("Recordings per class:")
display(selection["label"].value_counts().to_frame("recordings"))

print("Recordings per attack ('-' = bona fide):")
display(selection["attack"].value_counts().sort_index().to_frame("recordings"))

paths = [EVAL_AUDIO_DIR / f"{name}.flac" for name in selection["file_name"]]
missing = [path.name for path in paths if not path.exists()]
print("Files missing on disk:", missing if missing else "none")

### Step 2b – Save the selection and copy the original recordings

**What it does:** Saves the table of selected recordings to `results/prototype/selection.csv` and
copies the original FLAC files unchanged to `outputs/06_prototype_leading_silence_output/original/`.
Files that are already there are not copied again.

**Why:** `selection.csv` (small, committed to git) documents exactly which recordings were used.
The audio copies (not committed – the dataset licence does not allow sharing it, and the files
are large) are the input for the next steps and will later be sent to the pod.

In [ ]:
import shutil

selection.to_csv(RESULTS_DIR / "selection.csv", index=False)
print("Saved:", (RESULTS_DIR / "selection.csv").relative_to(REPO_ROOT))

ORIGINAL_DIR = OUTPUTS_DIR / "original"
ORIGINAL_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
for name in selection["file_name"]:
    target = ORIGINAL_DIR / f"{name}.flac"
    if not target.exists():
        shutil.copy2(EVAL_AUDIO_DIR / f"{name}.flac", target)
        copied += 1

in_folder = sorted(path.stem for path in ORIGINAL_DIR.glob("*.flac"))
print(f"Copied now: {copied} | files in {ORIGINAL_DIR.relative_to(REPO_ROOT)}: {len(in_folder)}")
if in_folder != sorted(selection["file_name"]):
    print("WARNING: the folder does not match the selection (e.g. files from another seed)")

### Step 3 – Remove the leading silence

**What it does:** For every threshold in `THRESHOLDS_DB` calls `trim_files`, which loads every
original recording, removes the silence from its beginning and saves the result under the same
file name in a separate folder per threshold, e.g.
`outputs/06_prototype_leading_silence_output/trimmed_-40dB/`. Then it shows how much silence was
removed, per threshold and class, checks that all files were written, and saves one table per
threshold, e.g. `results/prototype/trim_info_-40dB.csv`.

**Why:** The trimmed recordings are the second input for the model (next to the originals).
One folder per threshold means that trying another threshold never overwrites the
previous results. The table documents how much was cut from every file and with which settings.

In [ ]:
import pandas as pd

from xai_audio_deepfake_shortcuts_audit.prototype.trim_files import trim_files

trim_infos = []
for threshold_db in THRESHOLDS_DB:
    trim_infos.append(
        trim_files(
            selection,
            ORIGINAL_DIR,
            trimmed_dir(threshold_db),
            SAMPLE_RATE,
            FRAME_MS,
            HOP_MS,
            threshold_db,
        )
    )
trim_info = pd.concat(trim_infos, ignore_index=True)

print("Recordings per class:", selection["label"].value_counts().to_dict())

removed = trim_info.groupby(["threshold_db", "label"])["removed_s"].describe()
removed = removed.drop(columns="count").round(2)
print("Removed leading silence [s] by threshold and class:")
display(removed)

no_silence = trim_info[trim_info["removed_s"] == 0].groupby(["threshold_db", "label"]).size()
print("Recordings with no leading silence removed:", no_silence.to_dict())

for threshold_db in THRESHOLDS_DB:
    n_files = len(list(trimmed_dir(threshold_db).glob("*.flac")))
    shortest = trim_info.loc[trim_info["threshold_db"] == threshold_db, "trimmed_s"].min()
    name = trimmed_dir(threshold_db).name
    print(f"{name}: {n_files} files | shortest after trimming: {shortest:.2f} s")

for threshold_db, table in trim_info.groupby("threshold_db"):
    csv_path = RESULTS_DIR / f"trim_info_{threshold_db:+.0f}dB.csv"
    table.to_csv(csv_path, index=False)
    print("Saved:", csv_path.relative_to(REPO_ROOT))

### Step 4 – Listen and check

**What it does:** Picks up to 8 recordings – for bona fide and for spoof separately: the one with
the longest removed silence, the median one and the one with the shortest (measured at
`THRESHOLD_DB`), plus the one where the thresholds disagree the most (the largest difference
between the shortest and the longest cut). For each it draws the original waveform with one cut line per threshold and
shows the players: the original and the trimmed version for every threshold. The plots are
saved to `results/prototype/trim_plots/`.

**Why:** Numbers alone do not show whether the trimming is correct. Looking and listening checks
two things: the silence is gone, and the beginning of the first word is not cut off. Hearing the
thresholds next to each other shows which one cuts too much or leaves silence; the extreme
cases and the recordings where the thresholds disagree are where a wrong threshold shows
up first.

In [ ]:
# how much the cut moves between the thresholds: 0 = all thresholds cut at the same place
spread = trim_info.groupby("file_name")["removed_s"].agg(lambda s: s.max() - s.min())

reference = trim_info[trim_info["threshold_db"] == THRESHOLD_DB]
reference = reference.assign(spread_s=reference["file_name"].map(spread))

indices = []
for _, group in reference.groupby("label"):
    group = group.sort_values("removed_s", kind="stable")
    # longest, median and shortest removed silence in this class
    picks = [group.index[-1], group.index[len(group) // 2], group.index[0]]
    widest = group["spread_s"].idxmax()  # where the thresholds disagree the most
    if widest not in picks:
        picks.append(widest)
    indices += picks

columns = ["file_name", "label", "attack", "original_s", "removed_s", "spread_s"]
examples = reference.loc[indices, columns]
display(examples)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio

from xai_audio_deepfake_shortcuts_audit.audio.load_audio import load_audio
from xai_audio_deepfake_shortcuts_audit.prototype.plot_trim import plot_trim

PLOTS_DIR = RESULTS_DIR / "trim_plots"

for row in examples.itertuples():
    original = load_audio(ORIGINAL_DIR / f"{row.file_name}.flac", SAMPLE_RATE)
    per_threshold = trim_info[trim_info["file_name"] == row.file_name]
    cuts = {f"{r.threshold_db:.0f} dB": r.removed_s for r in per_threshold.itertuples()}
    kind = row.label if row.attack == "-" else f"{row.label} {row.attack}"
    out_png = PLOTS_DIR / f"{row.file_name}.png"

    fig = plot_trim(original, cuts, f"{row.file_name} | {kind}", SAMPLE_RATE, out_png)
    display(fig)
    plt.close(fig)

    print("original:")
    display(Audio(original, rate=SAMPLE_RATE))
    for r in per_threshold.itertuples():
        trimmed = load_audio(trimmed_dir(r.threshold_db) / f"{row.file_name}.flac", SAMPLE_RATE)
        print(f"trimmed at {r.threshold_db:.0f} dB (removed {r.removed_s:.2f} s):")
        display(Audio(trimmed, rate=SAMPLE_RATE))

#### Before and after – one recording in detail

**What it does:** Draws one of the examples at the chosen threshold `THRESHOLD_DB` in two panels:
the original with the removed part shaded, and below it the trimmed recording, i.e. exactly what
the model receives.

**Why:** The plot above compares the thresholds; this one shows the result of the chosen one.
Both panels use the same time scale, so it is easy to see that only the beginning was removed.

In [ ]:
from xai_audio_deepfake_shortcuts_audit.prototype.plot_trim_before_after import (
    plot_trim_before_after,
)

example = examples.iloc[0]
original = load_audio(ORIGINAL_DIR / f"{example.file_name}.flac", SAMPLE_RATE)
trimmed = load_audio(trimmed_dir(THRESHOLD_DB) / f"{example.file_name}.flac", SAMPLE_RATE)
title = f"{example.file_name} | {example.label} | trimmed at {THRESHOLD_DB:.0f} dB"
out_png = PLOTS_DIR / f"{example.file_name}_before_after_{THRESHOLD_DB:+.0f}dB.png"

fig = plot_trim_before_after(original, trimmed, example.removed_s, title, SAMPLE_RATE, out_png)
display(fig)
plt.close(fig)

#### Listening check – fill in after listening


| File | Class – why picked | −30 dB | −35 dB | −40 dB | Notes |
|---|---|---|---|---|---|
| LA_E_4502910 | bona fide – longest | 3.17 s: | 3.17 s: | 3.17 s: | |
| LA_E_1357107 | bona fide – median | 0.91 s: | 0.91 s: | 0.91 s: | |
| LA_E_6531800 | bona fide – shortest | 1.10 s: | 0.41 s: | 0.41 s: | |
| LA_E_9050668 | bona fide – thresholds disagree most | 2.32 s: | 0.69 s: | 0.18 s: | |
| LA_E_4431129 | spoof A17 – longest | 2.05 s: | 2.05 s: | 2.05 s: | |
| LA_E_9191204 | spoof A07 – median | 0.09 s: | 0.08 s: | 0.00 s: | |
| LA_E_1476397 | spoof A13 – shortest | 0.00 s: | 0.00 s: | 0.00 s: | |
| LA_E_3820815 | spoof A18 – thresholds disagree most | 2.13 s: | 0.53 s: | 0.00 s: | |

**Decision:** `THRESHOLD_DB = -30.0` dB.